# TS-GNN — Pipeline Completa (One-Click)

Lancia **tutta** la pipeline TS-GNN in sequenza automatica:
1. 🔧 Monta Drive, installa dipendenze, controlla GPU
2. ⬇️  Scarica tutti i dataset (BRCA, TCGA, JASPAR, STRING...)
3. 🧬 Genera embedding ESM-2 per gli alleli TP53
4. 🚀 Lancia pipeline completa: GRN → preprocessing → training → valutazione
5. 📊 Mostra risultati e figure
6. 💾 Salva tutto su Drive

**⚠️ Prima di runnare**: imposta runtime **A100 GPU + High-RAM** in *Runtime → Change runtime type*.

Poi clicca **Runtime → Run all** e aspetta.

---
## 🔧 STEP 1 — Setup: Drive, Path, Dipendenze, GPU

In [ ]:
import os, sys

# ── Mount Google Drive ────────────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    IN_COLAB = True
    print('Google Drive mounted')
except ImportError:
    IN_COLAB = False
    print('Not on Colab — local execution')

# ── Locate project ───────────────────────────────────────────────────────────
# CHANGE THIS if you placed ts-gnn in a different Drive folder:
DRIVE_ROOT   = '/content/drive/MyDrive'
PROJECT_ROOT = os.path.join(DRIVE_ROOT, 'ts-gnn')

if IN_COLAB:
    # Support both:
    #   A) folder uploaded directly  -> MyDrive/ts-gnn/
    #   B) zip uploaded              -> MyDrive/ts-gnn.zip (auto-extracted here)
    if not os.path.isdir(PROJECT_ROOT):
        zip_path = os.path.join(DRIVE_ROOT, 'ts-gnn.zip')
        if os.path.exists(zip_path):
            print(f'Found {zip_path} — extracting to {DRIVE_ROOT} ...')
            import zipfile
            with zipfile.ZipFile(zip_path) as zf:
                zf.extractall(DRIVE_ROOT)
            print('Extraction complete.')
        else:
            print('Project not found. Two options:')
            print('  A) Drag-and-drop the ts-gnn/ folder into MyDrive/ via drive.google.com')
            print('  B) Upload ts-gnn.zip to MyDrive/ and re-run this cell')
            raise FileNotFoundError('ts-gnn not found in Drive')
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if not os.path.isdir(os.path.join(PROJECT_ROOT, 'src')):
        PROJECT_ROOT = os.getcwd()

# ── Set up paths ─────────────────────────────────────────────────────────────
DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
src_path = os.path.join(PROJECT_ROOT, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
os.environ['TSGNN_DATA_DIR'] = DATA_DIR
os.makedirs(DATA_DIR, exist_ok=True)

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'DATA_DIR     = {DATA_DIR}')
print(f'src in path  = {os.path.isdir(src_path)}')


In [ ]:
print('Installing dependencies (2-5 min first time)...')
import sys, subprocess

# ── Core bio stack ────────────────────────────────────────────────────────
!pip install -q scanpy anndata harmonypy scrublet leidenalg scvelo GEOparse
!pip install -q fair-esm
!pip install -q decoupler
!pip install -q networkx tqdm wandb openpyxl pyyaml optuna

# ── scvi-tools (optional — gold-standard batch correction) ────────────────
try:
    import scvi
    print('scvi-tools already installed')
except Exception:
    try:
        !pip install -q scvi-tools
        import scvi  # noqa: F811
        print('scvi-tools installed')
    except Exception as _e:
        print(f'scvi-tools unavailable ({type(_e).__name__}) — Harmony fallback will be used')

# ── PyTorch Geometric (auto-detect torch + CUDA version) ─────────────────
import torch
TORCH_VER = torch.__version__.split('+')[0]
CUDA_VER  = torch.version.cuda.replace('.', '')[:3] if torch.cuda.is_available() else 'cpu'
print(f'Torch {TORCH_VER}, CUDA {CUDA_VER}')
!pip install -q torch-geometric
if CUDA_VER != 'cpu':
    import subprocess as _sp
    _r = _sp.run(
        ['pip', 'install', '-q', 'pyg-lib', 'torch-scatter', 'torch-sparse',
         '-f', f'https://data.pyg.org/whl/torch-{TORCH_VER}+cu{CUDA_VER}.html'],
        capture_output=True
    )
    if _r.returncode != 0:
        print('PyG compiled extensions not available — CPU fallback OK')
else:
    print('CPU-only runtime — skipping PyG CUDA extensions')

# ── Install ts-gnn (no-deps: we installed everything above) ──────────────
!pip install -q --no-deps -e "{PROJECT_ROOT}"

print('All dependencies installed')


In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

if device == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU rilevata: {gpu_name} ({gpu_mem:.0f} GB VRAM)')
    if 'A100' in gpu_name:
        print('   🚀 A100 — configurazione ottimale per questo progetto')
    elif 'T4' in gpu_name or 'V100' in gpu_name:
        print('   ⚠️  GPU non-A100: il training sarà più lento (~2-3x)')
else:
    print('❌ Nessuna GPU — cambia runtime in Runtime → Change runtime type')
    print('   Il training su CPU potrebbe richiedere ore.')

print(f'   PyTorch {torch.__version__} | Python {sys.version.split()[0]}')

---
## ⬇️  STEP 2 — Download Dataset

In [ ]:
import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(message)s',
    handlers=[logging.StreamHandler()]
)

print('Downloading datasets...')
print(f'   Destination: {DATA_DIR}')
print('   Estimated total: ~3-5 GB (STRING alone is 700 MB)')
print('─' * 60)

from tsgnn.data.download import (
    download_brca_scrna, download_tcga_brca, download_jaspar,
    download_string, download_string_aliases, download_targetgenereg,
    download_encode_chipseq, download_depmap, download_regnetwork
)
from tqdm.notebook import tqdm

download_steps = [
    (download_brca_scrna,   'BRCA scRNA-seq (GSE176078, GSE158508)'),
    (download_tcga_brca,    'TCGA BRCA TP53 mutations (cBioPortal API)'),
    (download_jaspar,       'JASPAR 2024 motifs'),
    (download_string,       'STRING v12 PPI'),
    (download_string_aliases, 'STRING aliases'),
    (download_targetgenereg,'TargetGeneReg 2.0 (p53 targets)'),
    (download_encode_chipseq,'ENCODE ChIP-seq (MDA-MB-231, T47D, MCF7)'),
    (download_depmap,       'DepMap CRISPR knockout'),
    (download_regnetwork,   'RegNetwork'),
]

errors = []
for fn, desc in tqdm(download_steps, desc='Downloading'):
    try:
        fn()
    except Exception as e:
        errors.append((desc, str(e)))
        print(f'  WARNING: {desc} failed: {e}')

if errors:
    print(f'
{len(errors)} download(s) failed — check warnings above.')
    print('Pipeline will raise an error for any missing required dataset.')
    print('Fix the download or place files manually in:', DATA_DIR)
else:
    print('
All downloads complete.')


In [ ]:
# Verifica stato download
checks = [
    ('raw/brca/GSE176078',                     'BRCA scRNA (Wu et al.)'),
    ('raw/brca/GSE158508',                     'BRCA scRNA+ATAC'),
    ('raw/tcga',                               'TCGA BRCA TP53 mutations'),
    ('external/jaspar',                        'JASPAR 2024 TF list'),
    ('external/string',                        'STRING v12 PPI'),
    ('external/regnetwork',                    'RegNetwork'),
    ('external/targetgenereg',                 'Fischer 2017 p53 targets'),
    ('external/depmap',                        'DepMap gene effects'),
    ('external/encode',                        'ENCODE ChIP-seq'),
    ('external/esm2_embeddings',               'ESM-2 embeddings'),
]

print('📁 Stato dataset:')
all_ok = True
for rel_path, desc in checks:
    full = os.path.join(DATA_DIR, rel_path)
    if os.path.isdir(full) and os.listdir(full):
        files = [f for f in os.listdir(full) if os.path.isfile(os.path.join(full, f))]
        mb = sum(os.path.getsize(os.path.join(full, f)) for f in files) / 1e6
        print(f'  ✅ {desc:<35s} {len(files)} file(s), {mb:.0f} MB')
    else:
        print(f'  ⚠️  {desc:<35s} NON TROVATO (userà dati sintetici)')
        all_ok = False

if all_ok:
    print('\n🎉 Tutti i dataset presenti!')
else:
    print('\n⚠️  Alcuni dataset mancanti — pipeline continua con fallback sintetici.')

---
## 🧬 STEP 3 — Embedding ESM-2 per Alleli TP53

In [ ]:
from tsgnn.data.allele import generate_esm2_embeddings, TP53_HOTSPOT_MUTATIONS
print('🧬 Generazione embedding ESM-2...')
print(f'   Alleli: {list(TP53_HOTSPOT_MUTATIONS.keys())}')
print('   (Primo avvio: scarica ESM-2 650M ~2.5GB e calcola embedding ~5 min su A100)')
print('   (Avvii successivi: carica da cache, ~5 sec)')
# Cache ESM-2 model to Drive so it survives Colab disconnects
import os
if IN_COLAB:
    os.environ['TORCH_HOME'] = os.path.join(DRIVE_ROOT, 'torch_cache')
    os.makedirs(os.environ['TORCH_HOME'], exist_ok=True)
esm_embeddings = generate_esm2_embeddings(device=device)
print(f'\n✅ Embedding pronti per {len(esm_embeddings)} alleli:')
for name, emb in esm_embeddings.items():
    print(f'   {name:8s}: shape={tuple(emb.shape)}, norm={emb.norm():.3f}')

---
## 🚀 STEP 4 — Pipeline Completa: GRN → Preprocessing → Training → Valutazione

In [ ]:
# ── Patch: tutti i fix fp16/autocast in sheaf_vectorized.py ──────────────
import glob as _glob, os as _os

_path = PROJECT_ROOT + '/src/tsgnn/model/sheaf_vectorized.py'
with open(_path, encoding='utf-8') as f:
    _src = f.read()

_patches = [
    # 1: L_flat dtype (VectorizedSheafDiffusion)
    ('torch.zeros(Nd * Nd, device=dev, dtype=maps.dtype)',
     'torch.zeros(Nd * Nd, device=dev, dtype=off_diag.dtype)'),
    # 2: D_blocks dtype (NormalizedVectorizedSheafDiffusion)
    ('torch.zeros(N, d, d, device=maps.device, dtype=maps.dtype)',
     'torch.zeros(N, d, d, device=maps.device, dtype=FTF_src.dtype)'),
    # 3: eigh fp16 — cast D_blocks to float
    ('eigenvalues, eigenvectors = torch.linalg.eigh(D_blocks)',
     'eigenvalues, eigenvectors = torch.linalg.eigh(D_blocks.float())'),
    # 4: eigvalsh in spectral_gap
    ('eigenvalues = torch.linalg.eigvalsh(L)\n        # Sort and get gap',
     'eigenvalues = torch.linalg.eigvalsh(L.float())\n        # Sort and get gap'),
    # 5: eigvalsh in condition_number
    ('eigenvalues = torch.linalg.eigvalsh(L)\n        sorted_eigs = eigenvalues.sort().values\n        lambda_2',
     'eigenvalues = torch.linalg.eigvalsh(L.float())\n        sorted_eigs = eigenvalues.sort().values\n        lambda_2'),
    # 6: eigvalsh in verify_laplacian_properties
    ('eigs = torch.linalg.eigvalsh(L).sort().values',
     'eigs = torch.linalg.eigvalsh(L.float()).sort().values'),
]

_applied = 0
_already = 0
for old, new in _patches:
    if old in _src:
        _src = _src.replace(old, new, 1)
        _applied += 1
    elif new in _src:
        _already += 1
    else:
        print(f'⚠️  Pattern non trovato: {old[:60]}...')

with open(_path, 'w', encoding='utf-8') as f:
    f.write(_src)

print(f'✅ {_applied} patch applicate, {_already} già presenti')

# Elimina __pycache__
_removed = 0
for _pyc in _glob.glob(PROJECT_ROOT + '/src/tsgnn/model/__pycache__/sheaf_vectorized*.pyc'):
    _os.remove(_pyc)
    _removed += 1
print(f'🗑️  {_removed} file .pyc rimossi')

In [ ]:
# --- Checkpoint Resume ---
# Automatically resume training if a checkpoint exists on Drive.
# This lets you continue a Colab session after disconnect without losing progress.
from pathlib import Path

checkpoint_dir = Path(PROJECT_ROOT) / "checkpoints"
best_ckpt = checkpoint_dir / "best.pt"

if best_ckpt.exists():
    import torch
    ckpt = torch.load(str(best_ckpt), map_location="cpu", weights_only=False)
    print(f"Found checkpoint: epoch={ckpt['epoch']}, val_loss={ckpt['val_loss']:.6f}")
    print("Training will resume from this checkpoint automatically.")
else:
    print("No checkpoint found -- training will start from scratch.")


In [ ]:
import subprocess, glob, sys

config_path     = os.path.join(PROJECT_ROOT, 'configs', 'default.yaml')
pipeline_script = os.path.join(PROJECT_ROOT, 'scripts', 'run_pipeline.py')

print('Starting full pipeline...')
print(f'   Script : {pipeline_script}')
print(f'   Config : {config_path}')
print(f'   Device : {device}')
print('-' * 60)

# Pass TSGNN_DATA_DIR so subprocess finds the data
env = os.environ.copy()
env['TSGNN_DATA_DIR'] = DATA_DIR

cmd = [sys.executable, pipeline_script, '--config', config_path, '--skip-download']
print('Training from scratch (or resuming via trainer checkpoint logic)')

# Stream output line by line so you see progress in real time
import subprocess as _sp
proc = _sp.Popen(cmd, stdout=_sp.PIPE, stderr=_sp.STDOUT,
                 text=True, bufsize=1, env=env)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

print('-' * 60)
if proc.returncode == 0:
    print('Pipeline completed successfully!')
else:
    print(f'Pipeline exited with code {proc.returncode}')
    print('Scroll up to find the error.')


---
## 📊 STEP 5 — Risultati e Figure

In [ ]:
import glob as glob_module

checkpoint_dir = os.path.join(PROJECT_ROOT, 'checkpoints')
checkpoints    = sorted(glob_module.glob(os.path.join(checkpoint_dir, '*.pt')))

if checkpoints:
    latest_ckpt = checkpoints[-1]
    ckpt = torch.load(latest_ckpt, map_location='cpu')
    print(f'📂 Checkpoint: {os.path.basename(latest_ckpt)}')
    print(f'   Epoch:     {ckpt.get("epoch", "n/a")}')
    print(f'   Val loss:  {ckpt.get("val_loss", "n/a")}')
    print(f'   Train loss:{ckpt.get("train_loss", "n/a")}')
else:
    print(f'⚠️  Nessun checkpoint in {checkpoint_dir}')
    ckpt = None

In [ ]:
from IPython.display import Image, display

fig_dir = os.path.join(PROJECT_ROOT, 'figures')
figs    = sorted(glob_module.glob(os.path.join(fig_dir, '*.png')))

if figs:
    print(f'📊 {len(figs)} figure trovate:')
    for fp in figs:
        print(f'  → {os.path.basename(fp)}')
        display(Image(fp, width=700))
else:
    print('ℹ️  Nessuna figura PNG trovata in', fig_dir)
    print('   Le figure vengono generate durante la fase di visualizzazione del pipeline.')

In [ ]:
# Quick checkpoint sanity check (no synthetic data — just verifies model loads correctly)
if ckpt is not None:
    import yaml
    from tsgnn.model.tsgnn import TSGNN

    with open(config_path) as f:
        cfg = yaml.safe_load(f)

    N = cfg.get('model', {}).get('input_dim', 500)
    E = 1000  # placeholder edge count for model init only
    d = cfg.get('model', {}).get('stalk_dim', 4)

    edge_index = torch.zeros(2, E, dtype=torch.long)
    model = TSGNN(
        num_nodes=N, num_edges=E, stalk_dim=d, input_dim=N,
        esm_dim=1280, conditioning_dim=128,
        edge_index=edge_index, num_diffusion_steps=3,
    )

    if 'model_state_dict' in ckpt:
        try:
            missing, unexpected = model.load_state_dict(ckpt['model_state_dict'], strict=False)
            print(f'Weights loaded from checkpoint')
            if missing:    print(f'  Missing keys   : {len(missing)}')
            if unexpected: print(f'  Unexpected keys: {len(unexpected)}')
        except RuntimeError as e:
            print(f'⚠️  Checkpoint incompatibile con il modello attuale (shape mismatch).')
            print(f'   Causa: il checkpoint è stato salvato con architettura diversa.')
            print(f'   Soluzione: elimina il checkpoint e ri-runna il training.')
            print(f'   Dettaglio: {str(e)[:200]}')
            print(f'\n   Procedo con modello inizializzato da zero.')

    model.eval().to(device)

    # Verify forward pass runs (shape check only, no real data needed)
    with torch.no_grad():
        dummy_x   = torch.zeros(3, N, N, device=device)   # K=3 steps
        dummy_emb = torch.zeros(1280, device=device)
        preds, _, _ = model(dummy_x, dummy_emb)
    print(f'Model forward pass OK: {len(preds)} steps, each {tuple(preds[0].shape)}')
    print(f'Run the full pipeline to evaluate on real BRCA data.')
else:
    print('No checkpoint available — run pipeline first.')

---
## 💾 STEP 6 — Salva Risultati su Drive

In [ ]:
import shutil, datetime

timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')

if IN_COLAB:
    save_dir = f'/content/drive/MyDrive/tsgnn_results_{timestamp}'
    os.makedirs(save_dir, exist_ok=True)

    # Checkpoints
    if os.path.isdir(checkpoint_dir) and os.listdir(checkpoint_dir):
        shutil.copytree(checkpoint_dir, os.path.join(save_dir, 'checkpoints'),
                        dirs_exist_ok=True)
        print(f'✅ Checkpoints → {save_dir}/checkpoints/')

    # Figure
    if os.path.isdir(fig_dir) and os.listdir(fig_dir):
        shutil.copytree(fig_dir, os.path.join(save_dir, 'figures'),
                        dirs_exist_ok=True)
        print(f'✅ Figure       → {save_dir}/figures/')

    # Embedding ESM-2
    emb_dir = os.path.join(save_dir, 'esm2_embeddings')
    os.makedirs(emb_dir, exist_ok=True)
    for name, emb in esm_embeddings.items():
        torch.save(emb, os.path.join(emb_dir, f'{name}.pt'))
    print(f'✅ ESM-2 emb.   → {emb_dir}/')

    print(f'\n🎉 Tutto salvato in: {save_dir}')
else:
    print(f'ℹ️  Locale: risultati già in {checkpoint_dir} e {fig_dir}')

---
## 🔬 STEP 7 (opzionale) — Ablation Study

Decommenta la cella sotto per runnare l'ablation study completo (~1-2h su A100).

In [ ]:
# ablation_script = os.path.join(PROJECT_ROOT, 'scripts', 'run_ablations.py')
# !python "{ablation_script}" --config "{config_path}"

print('ℹ️  Ablation study non avviato (decommenta sopra per attivarlo).')

---
## ✅ Riepilogo Pipeline

| Step | Operazione | Note |
|------|-----------|------|
| 1 | Setup Drive + dipendenze + GPU | Automatico |
| 2 | Download dataset (~3-5 GB) | Automatico con fallback sintetici |
| 3 | ESM-2 embeddings TP53 alleli | Cache automatica dopo primo run |
| 4 | GRN + preprocessing + training + eval | `scripts/run_pipeline.py` |
| 5 | Mostra risultati e figure | Automatico |
| 6 | Salva su Drive | Automatico |
| 7 | Ablation study | Opzionale |

Per modificare i parametri → edita `configs/default.yaml`.

Per dettagli architettura → vedi `PROJECT_CONTEXT.md` nella root del progetto.